In [1]:
# Notebook 04 - Commit History & Longitudinal Analysis
# Author: Vidhumol Pandippilly Antony | ST86889
# Purpose: Collect commit history and longitudinal debt trends

!pip install PyGithub pydriller pandas -q

import pandas as pd
import time
from github import Github
from google.colab import userdata
from datetime import datetime
from collections import defaultdict

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
g = Github(GITHUB_TOKEN)

print(f"Connected as: {g.get_user().login}")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.7/449.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
Connected as: vidhumol
Started: 2026-05-06 21:50:10


/tmp/ipykernel_18674/1690533047.py:15: DeprecationWarning: Argument login_or_token is deprecated, please use auth=github.Auth.Token(...) instead
  g = Github(GITHUB_TOKEN)


In [2]:
repositories = [
    {"repo": "keras-team/keras",                "category": "Deep Learning"},
    {"repo": "fastai/fastai",                   "category": "Deep Learning"},
    {"repo": "pytorch/examples",                "category": "Deep Learning"},
    {"repo": "Lightning-AI/pytorch-lightning",  "category": "Deep Learning"},
    {"repo": "huggingface/datasets",            "category": "NLP"},
    {"repo": "explosion/spaCy",                 "category": "NLP"},
    {"repo": "facebookresearch/fairseq",        "category": "NLP"},
    {"repo": "allenai/allennlp",                "category": "NLP"},
    {"repo": "facebookresearch/detectron2",     "category": "Computer Vision"},
    {"repo": "ultralytics/yolov5",              "category": "Computer Vision"},
    {"repo": "open-mmlab/mmdetection",          "category": "Computer Vision"},
    {"repo": "rwightman/pytorch-image-models",  "category": "Computer Vision"},
    {"repo": "mlflow/mlflow",                   "category": "MLOps"},
    {"repo": "iterative/dvc",                   "category": "MLOps"},
    {"repo": "PrefectHQ/prefect",               "category": "MLOps"},
    {"repo": "bentoml/BentoML",                 "category": "MLOps"},
    {"repo": "optuna/optuna",                   "category": "AutoML/RL"},
    {"repo": "openai/gym",                      "category": "AutoML/RL"},
    {"repo": "microsoft/nni",                   "category": "AutoML/RL"},
    {"repo": "automl/auto-sklearn",             "category": "AutoML/RL"},
]
print(f"{len(repositories)} repositories ready!")

20 repositories ready!


In [3]:
def collect_commit_history(repo_info, github_obj):
    """Collect monthly commit frequency and issue data"""
    repo_name = repo_info["repo"]
    print(f"\nCollecting: {repo_name}")

    monthly_data = []

    try:
        repo = github_obj.get_repo(repo_name)

        # Get commits - last 500 max
        commits = repo.get_commits()
        commit_list = []

        count = 0
        for commit in commits:
            try:
                commit_list.append({
                    "date":    commit.commit.author.date,
                    "message": commit.commit.message[:100],
                    "author":  commit.commit.author.name,
                })
                count += 1
                if count >= 500:
                    break
            except:
                pass

        print(f"   Collected {len(commit_list)} commits")

        # Group commits by month
        monthly_counts = defaultdict(int)
        for c in commit_list:
            month_key = c["date"].strftime("%Y-%m")
            monthly_counts[month_key] += 1

        # Build monthly records
        for month, count in sorted(monthly_counts.items()):
            monthly_data.append({
                "repo_name": repo_name,
                "category":  repo_info["category"],
                "month":     month,
                "commit_count": count,
            })

        print(f"   Monthly data: {len(monthly_data)} months covered")
        time.sleep(2)

    except Exception as e:
        print(f"   Error: {e}")

    return monthly_data

print("Commit history function ready!")

Commit history function ready!


In [4]:
def collect_issue_data(repo_info, github_obj):
    """Collect issue resolution times"""
    repo_name = repo_info["repo"]
    print(f"\nCollecting issues: {repo_name}")

    issue_data = []

    try:
        repo = github_obj.get_repo(repo_name)

        # Get closed issues - max 100
        issues = repo.get_issues(state='closed')

        count = 0
        for issue in issues:
            try:
                if issue.closed_at and issue.created_at:
                    resolution_days = (issue.closed_at - issue.created_at).days
                    issue_data.append({
                        "repo_name":       repo_name,
                        "category":        repo_info["category"],
                        "issue_number":    issue.number,
                        "created_at":      issue.created_at.strftime("%Y-%m-%d"),
                        "closed_at":       issue.closed_at.strftime("%Y-%m-%d"),
                        "resolution_days": resolution_days,
                        "labels":          ",".join([l.name for l in issue.labels]),
                    })
                count += 1
                if count >= 100:
                    break
            except:
                pass

        print(f"   Collected {len(issue_data)} closed issues")
        time.sleep(2)

    except Exception as e:
        print(f"   Error: {e}")

    return issue_data

print("Issue collection function ready!")

Issue collection function ready!


In [5]:
print("Starting commit history collection...\n")

all_commits = []
all_issues = []

for i, repo_info in enumerate(repositories, 1):
    print(f"[{i}/20] {repo_info['repo']}")

    # Collect commits
    commits = collect_commit_history(repo_info, g)
    all_commits.extend(commits)

    # Collect issues
    issues = collect_issue_data(repo_info, g)
    all_issues.extend(issues)

    time.sleep(1)

print(f"\n{'='*50}")
print(f"Collection Complete!")
print(f"Monthly commit records: {len(all_commits)}")
print(f"Issue records: {len(all_issues)}")

Starting commit history collection...

[1/20] keras-team/keras

Collecting: keras-team/keras
   Collected 500 commits
   Monthly data: 5 months covered

   Collected 100 closed issues
[2/20] fastai/fastai

Collecting: fastai/fastai
   Collected 500 commits
   Monthly data: 43 months covered

   Collected 100 closed issues
[3/20] pytorch/examples

Collecting: pytorch/examples
   Collected 500 commits
   Monthly data: 76 months covered

   Collected 100 closed issues
[4/20] Lightning-AI/pytorch-lightning

Collecting: Lightning-AI/pytorch-lightning
   Collected 500 commits
   Monthly data: 14 months covered

   Collected 100 closed issues
[5/20] huggingface/datasets

Collecting: huggingface/datasets
   Collected 500 commits
   Monthly data: 26 months covered

   Collected 100 closed issues
[6/20] explosion/spaCy

Collecting: explosion/spaCy
   Collected 500 commits
   Monthly data: 30 months covered

   Collected 100 closed issues
[7/20] facebookresearch/fairseq

Collecting: facebookresea

Following Github server redirection from /repos/rwightman/pytorch-image-models to /repositories/168799526
INFO:github.Requester:Following Github server redirection from /repos/rwightman/pytorch-image-models to /repositories/168799526


[12/20] rwightman/pytorch-image-models

Collecting: rwightman/pytorch-image-models
   Collected 500 commits
   Monthly data: 16 months covered


Following Github server redirection from /repos/rwightman/pytorch-image-models to /repositories/168799526
INFO:github.Requester:Following Github server redirection from /repos/rwightman/pytorch-image-models to /repositories/168799526



   Collected 100 closed issues
[13/20] mlflow/mlflow

Collecting: mlflow/mlflow
   Collected 500 commits
   Monthly data: 3 months covered

   Collected 100 closed issues


Following Github server redirection from /repos/iterative/dvc to /repositories/83878269
INFO:github.Requester:Following Github server redirection from /repos/iterative/dvc to /repositories/83878269


[14/20] iterative/dvc

Collecting: iterative/dvc
   Collected 500 commits
   Monthly data: 30 months covered


Following Github server redirection from /repos/iterative/dvc to /repositories/83878269
INFO:github.Requester:Following Github server redirection from /repos/iterative/dvc to /repositories/83878269



   Collected 100 closed issues
[15/20] PrefectHQ/prefect

Collecting: PrefectHQ/prefect
   Collected 500 commits
   Monthly data: 3 months covered

   Collected 100 closed issues
[16/20] bentoml/BentoML

Collecting: bentoml/BentoML
   Collected 500 commits
   Monthly data: 22 months covered

   Collected 100 closed issues
[17/20] optuna/optuna

Collecting: optuna/optuna
   Collected 500 commits
   Monthly data: 7 months covered

   Collected 100 closed issues
[18/20] openai/gym

Collecting: openai/gym
   Collected 500 commits
   Monthly data: 18 months covered

   Collected 100 closed issues
[19/20] microsoft/nni

Collecting: microsoft/nni
   Collected 500 commits
   Monthly data: 20 months covered

   Collected 100 closed issues
[20/20] automl/auto-sklearn

Collecting: automl/auto-sklearn
   Collected 500 commits
   Monthly data: 34 months covered

   Collected 100 closed issues

Collection Complete!
Monthly commit records: 534
Issue records: 2000


In [6]:
df_commits = pd.DataFrame(all_commits)
df_issues = pd.DataFrame(all_issues)

print("Commit History Summary:")
print(df_commits.groupby('repo_name')['commit_count'].sum().sort_values(ascending=False).to_string())

print("\nIssue Resolution Summary (avg days):")
print(df_issues.groupby('repo_name')['resolution_days'].mean().round(1).sort_values(ascending=False).to_string())

df_commits.to_csv('commit_history.csv', index=False)
df_issues.to_csv('issue_data.csv', index=False)

print(f"\nSaved commit_history.csv and issue_data.csv!")

Commit History Summary:
repo_name
Lightning-AI/pytorch-lightning    500
PrefectHQ/prefect                 500
allenai/allennlp                  500
automl/auto-sklearn               500
bentoml/BentoML                   500
explosion/spaCy                   500
facebookresearch/detectron2       500
facebookresearch/fairseq          500
fastai/fastai                     500
huggingface/datasets              500
iterative/dvc                     500
keras-team/keras                  500
microsoft/nni                     500
mlflow/mlflow                     500
open-mmlab/mmdetection            500
openai/gym                        500
optuna/optuna                     500
pytorch/examples                  500
rwightman/pytorch-image-models    500
ultralytics/yolov5                500

Issue Resolution Summary (avg days):
repo_name
explosion/spaCy                   84.1
automl/auto-sklearn               64.1
fastai/fastai                     46.9
pytorch/examples                  38.2
fa